# Integración y evaluación final del sistema de aves

Integramos **RF-DETR Nano E5-v2** y **Swin-T**, comprobamos su funcionamiento en `validation`, congelamos la configuración y efectuamos las tres evaluaciones independientes sobre `test`. Finalmente exportamos un paquete reutilizable para el notebook de demostración.


## Preparación


In [1]:
%pip -q install "rfdetr==1.12.0" "kagglehub==1.0.3" soundfile scipy scikit-learn pandas matplotlib torchvision pillow

In [2]:
from pathlib import Path
from io import BytesIO
import hashlib, json, shutil, tarfile, sys
import kagglehub
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, f1_score
from google.colab import drive
from tqdm.auto import tqdm

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Selecciona GPU (T4 o superior) en Colab."
BASE = Path("/content/drive/MyDrive/Proyecto_Aves")
TRABAJO = Path("/content/aves_integracion")
RESULTADOS = BASE / "experiments/final/Evaluacion_Test_v1"
EXPORT = BASE / "Sistema_Aves_v1"
DIR_DET = BASE / "experiments/detector/E5_RFDETR_Negativas/entrenamiento_v2"
DIR_SWIN = BASE / "experiments/classifier/C2_Swin_T/final_v1"
KAGGLE_ORIGEN = "santiagocorrea802/dataset-proyecto"
KAGGLE_CLAS = "santiagocorrea802/clasificador-aves-espectrogramas-v1/versions/1"
for d in (TRABAJO, RESULTADOS, EXPORT, RESULTADOS / "predicciones_audio", RESULTADOS / "predicciones_clasificador"):
    d.mkdir(parents=True, exist_ok=True)
ARCHIVOS = [DIR_DET / "checkpoint_best_total.pth", DIR_DET / "configuracion.json",
            DIR_SWIN / "checkpoint_best.pth", BASE / "splits/recording_split_seed0.csv"]
for ruta in ARCHIVOS:
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta en Drive: {ruta}")

def descargar(dataset, ruta):
    return Path(kagglehub.dataset_download(dataset, path=ruta, output_dir=str(TRABAJO / "kaggle")))

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

splits = pd.read_csv(ARCHIVOS[3])
assert splits.recording_uid.is_unique
assert splits.split.value_counts().to_dict() == {"train": 707, "validation": 152, "test": 152}
print("GPU:", torch.cuda.get_device_name(0), "| Splits: correctos")

Mounted at /content/drive
GPU: Tesla T4 | Splits: correctos


## Sistema de inferencia

Implementación: audio de 32 kHz -> ventanas de 8 s con salto de 4 s -> RF-DETR -> fusión NMS -> espectrogramas de las detecciones -> Swin-T. Para mantener comparabilidad con los experimentos anteriores, usamos **confianza mínima 0,05**, **NMS IoU 0,5** y **umbral operativo 0,50**, fijados antes de abrir `test`.

In [3]:
%%writefile /content/sistema_aves.py
from __future__ import annotations

from pathlib import Path
from contextlib import nullcontext
import json

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from PIL import Image
from scipy.signal import stft
from torchvision import transforms
from torchvision.models import swin_t
from rfdetr import RFDETRNano

FS, DURACION_VENTANA, SALTO = 32000, 8, 4
COLUMNAS = ["recording_uid", "t_min", "t_max", "f_min", "f_max",
            "score_detector", "pred_code", "pred_label", "score_clasificador"]


def inicios_ventanas(n_muestras: int) -> list[int]:
    tam, salto = FS * DURACION_VENTANA, FS * SALTO
    if n_muestras <= tam:
        return [0]
    ultimo = n_muestras - tam
    inicios = list(range(0, ultimo + 1, salto))
    if inicios[-1] != ultimo:
        if len(inicios) > 1 and ultimo - inicios[-1] <= salto / 2 and ultimo - inicios[-2] <= tam:
            inicios[-1] = ultimo
        else:
            inicios.append(ultimo)
    return inicios


def espectrograma(senal: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    frecuencias, _, z = stft(senal, fs=FS, nperseg=1024, noverlap=768, boundary=None)
    return frecuencias, z


def imagen_detector(senal: np.ndarray) -> Image.Image:
    _, z = espectrograma(senal)
    db = 20 * np.log10(np.maximum(np.abs(z), 1e-8))
    vmax = np.percentile(db, 99)
    gris = np.flipud(np.clip((db - (vmax - 80)) / 80, 0, 1))
    return Image.fromarray(np.rint(gris * 255).astype(np.uint8), mode="L").convert("RGB")


def imagen_clasificador(senal: np.ndarray, f_min: float, f_max: float) -> Image.Image:
    inicio_hz = min(max((f_min + f_max) / 2 - 4000, 0), 8000)
    frecuencias, z = espectrograma(senal)
    banda = (frecuencias >= inicio_hz) & (frecuencias <= inicio_hz + 8000)
    if banda.sum() < 2:
        raise ValueError("Banda de frecuencia inválida")
    db = 20 * np.log10(np.maximum(np.abs(z[banda]), 1e-8))
    vmax = np.percentile(db, 99)
    gris = np.flipud(np.clip((db - (vmax - 80)) / 80, 0, 1))
    return Image.fromarray(np.rint(gris * 255).astype(np.uint8), mode="L").resize(
        (224, 224), resample=Image.Resampling.BILINEAR)


def iou(caja: np.ndarray, otras: np.ndarray) -> np.ndarray:
    if not len(otras):
        return np.empty(0, dtype=np.float32)
    inter = np.maximum(np.minimum(caja[2:], otras[:, 2:]) -
                       np.maximum(caja[:2], otras[:, :2]), 0).prod(axis=1)
    area_a = np.maximum(caja[2:] - caja[:2], 0).prod()
    area_b = np.maximum(otras[:, 2:] - otras[:, :2], 0).prod(axis=1)
    union = area_a + area_b - inter
    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)


def nms(cajas: np.ndarray, umbral: float = 0.5) -> np.ndarray:
    if not len(cajas):
        return cajas.reshape(0, 5)
    orden = np.argsort(-cajas[:, 4], kind="stable")
    seleccion = []
    while len(orden):
        i = int(orden[0])
        seleccion.append(i)
        resto = orden[1:]
        orden = resto[iou(cajas[i, :4], cajas[resto, :4]) < umbral]
    return cajas[seleccion]


class SistemaAves:
    def __init__(self, ruta_detector, config_detector, ruta_swin, codigos: dict[int, str]):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        config = json.loads(Path(config_detector).read_text(encoding="utf-8"))
        self.detector = RFDETRNano(
            pretrain_weights=str(ruta_detector), num_classes=1,
            resolution=int(config["resolution"]),
        )
        self.swin = swin_t(weights=None)
        self.swin.head = nn.Linear(self.swin.head.in_features, len(codigos))
        estado = torch.load(ruta_swin, map_location="cpu", weights_only=True)
        self.swin.load_state_dict(estado["model_state_dict"])
        self.swin.to(self.device).eval()
        self.codigos = codigos
        self.transformar = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    @classmethod
    def desde_bundle(cls, carpeta):
        carpeta = Path(carpeta)
        mapa = json.loads((carpeta / "class_mapping.json").read_text(encoding="utf-8"))
        codigos = {int(x["id"]): x["name"] for x in mapa}
        return cls(carpeta / "modelo_detector.pth", carpeta / "config_detector.json",
                   carpeta / "modelo_clasificador.pth", codigos)

    @torch.inference_mode()
    def clasificar_imagenes(self, imagenes: list[Image.Image]) -> tuple[list[int], list[float]]:
        x = torch.stack([self.transformar(im.convert("RGB")) for im in imagenes]).to(self.device)
        contexto = torch.autocast("cuda", dtype=torch.float16) if self.device.type == "cuda" else nullcontext()
        with contexto:
            probabilidades = self.swin(x).softmax(dim=1)
        confianza, indice = probabilidades.max(dim=1)
        return indice.cpu().tolist(), confianza.float().cpu().tolist()

    def predecir_audio(self, ruta_audio, uid: str, conf_min: float = 0.05,
                      conf_operativa: float = 0.50) -> pd.DataFrame:
        candidatos = []
        with sf.SoundFile(ruta_audio) as audio:
            if audio.samplerate != FS or audio.channels != 1:
                raise ValueError(f"Formato de audio incompatible: {ruta_audio}")
            duracion = audio.frames / FS
            inicios = inicios_ventanas(audio.frames)
            for k in range(0, len(inicios), 2):
                imagenes = []
                for inicio in inicios[k:k + 2]:
                    audio.seek(inicio)
                    senal = audio.read(FS * DURACION_VENTANA, dtype="float32")
                    senal = np.pad(senal, (0, FS * DURACION_VENTANA - len(senal)))
                    imagenes.append(imagen_detector(senal))
                salidas = self.detector.predict(imagenes, threshold=conf_min)
                for inicio, imagen, salida in zip(inicios[k:k + 2], imagenes, salidas):
                    ancho, alto = imagen.size
                    escala = np.array([ancho, alto, ancho, alto], dtype=np.float32)
                    cajas = np.asarray(salida.xyxy, dtype=np.float32).reshape(-1, 4) / escala
                    scores = np.asarray(salida.confidence, dtype=np.float32).reshape(-1)
                    for caja, score in zip(cajas, scores):
                        t0 = max(0., inicio / FS + float(caja[0]) * DURACION_VENTANA)
                        t1 = min(duracion, inicio / FS + float(caja[2]) * DURACION_VENTANA)
                        y0, y1 = float(np.clip(caja[1], 0, 1)), float(np.clip(caja[3], 0, 1))
                        if t1 > t0 and y1 > y0:
                            candidatos.append([t0, y0, t1, y1, float(score)])
            cajas = nms(np.asarray(candidatos, dtype=np.float32).reshape(-1, 5))
            salida = pd.DataFrame(columns=COLUMNAS)
            if not len(cajas):
                return salida
            filas = []
            for t0, y0, t1, y1, score in cajas:
                filas.append({"recording_uid": uid, "t_min": float(t0), "t_max": float(t1),
                              "f_min": float((1 - y1) * 16000), "f_max": float((1 - y0) * 16000),
                              "score_detector": float(score), "pred_code": np.nan,
                              "pred_label": None, "score_clasificador": np.nan})
            salida = pd.DataFrame(filas, columns=COLUMNAS)
            indices = salida.index[salida.score_detector >= conf_operativa].tolist()
            for k in range(0, len(indices), 16):
                grupo = indices[k:k + 16]
                imagenes = []
                for idx in grupo:
                    r = salida.loc[idx]
                    centro = (float(r.t_min) + float(r.t_max)) / 2
                    comienzo = min(max(centro - 4, 0), max(duracion - 8, 0))
                    audio.seek(min(round(comienzo * FS), audio.frames))
                    senal = audio.read(FS * DURACION_VENTANA, dtype="float32")
                    senal = np.pad(senal, (0, FS * DURACION_VENTANA - len(senal)))
                    imagenes.append(imagen_clasificador(senal, float(r.f_min), float(r.f_max)))
                codigos, confianza = self.clasificar_imagenes(imagenes)
                for idx, codigo, conf in zip(grupo, codigos, confianza):
                    salida.at[idx, "pred_code"] = int(codigo)
                    salida.at[idx, "pred_label"] = self.codigos[int(codigo)]
                    salida.at[idx, "score_clasificador"] = float(conf)
        return salida

Writing /content/sistema_aves.py


In [4]:
sys.path.insert(0, "/content")
from sistema_aves import SistemaAves

mapa = json.loads(descargar(KAGGLE_CLAS, "class_mapping.json").read_text(encoding="utf-8"))
codigos = {int(x["id"]): x["name"] for x in mapa}
preproceso = json.loads(descargar(KAGGLE_CLAS, "preprocessing_config.json").read_text(encoding="utf-8"))
assert len(codigos) == 31 and codigos[0] == "OTHER_SPECIES"
assert preproceso["sample_rate"] == 32000 and preproceso["window_s"] == 8
assert preproceso["freq_crop_width_hz"] == 8000 and preproceso["png_size"] == [224, 224]
sistema = SistemaAves(ARCHIVOS[0], ARCHIVOS[1], ARCHIVOS[2], codigos)
print("Sistema cargado: RF-DETR E5-v2 + Swin-T")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Using Colab cache for faster access to the 'clasificador-aves-espectrogramas-v1' dataset.
Using Colab cache for faster access to the 'clasificador-aves-espectrogramas-v1' dataset.
Sistema cargado: RF-DETR E5-v2 + Swin-T


## Verificación en validation

Ejecutamos el pipeline sobre dos grabaciones cortas de `validation`. Es una **prueba funcional**, no una nueva estimación estadística. Ninguna anotación participa como entrada de los modelos. Los umbrales operativos se conservan según las evaluaciones anteriores.

In [5]:
recordings = pd.read_csv(descargar(KAGGLE_ORIGEN, "DATASET_PROYECTO/metadata/recordings_manifest.csv"))
assert {"recording_uid", "normalized_path", "duration_sec"} <= set(recordings)
catalogo = splits.merge(recordings[["recording_uid", "normalized_path", "duration_sec"]],
                       on="recording_uid", validate="one_to_one")

def ruta_audio(fila):
    relativa = str(fila.normalized_path).replace("\\", "/").lstrip("/")
    if not relativa.startswith("DATASET_PROYECTO/"):
        relativa = "DATASET_PROYECTO/" + relativa
    return descargar(KAGGLE_ORIGEN, relativa)

eventos_val = pd.read_csv(descargar(KAGGLE_ORIGEN, "DATASET_PROYECTO/metadata/classifier_manifest.csv"))
ids_val = set(eventos_val.recording_uid) & set(catalogo.loc[catalogo.split.eq("validation"), "recording_uid"])
validacion = catalogo.loc[catalogo.recording_uid.isin(ids_val)].nsmallest(2, "duration_sec")
assert len(validacion) == 2
clasificaciones = 0
for fila in validacion.itertuples(index=False):
    resultado = sistema.predecir_audio(ruta_audio(fila), fila.recording_uid)
    n = int(resultado.pred_code.notna().sum())
    clasificaciones += n
    print(f"{fila.recording_uid}: {len(resultado)} cajas; {n} clasificadas")
    assert set(resultado.columns) == set(__import__("sistema_aves").COLUMNAS)
if clasificaciones == 0:
    # Solo para verificar el paso de Swin-T, sin modificar el protocolo.
    import soundfile as sf
    from sistema_aves import imagen_clasificador, FS
    evento = eventos_val.loc[eventos_val.recording_uid.eq(validacion.iloc[0].recording_uid)].iloc[0]
    with sf.SoundFile(ruta_audio(validacion.iloc[0])) as audio:
        inicio = min(max((evento.t_min + evento.t_max) / 2 - 4, 0), max(len(audio) / FS - 8, 0))
        audio.seek(round(inicio * FS))
        senal = audio.read(FS * 8, dtype="float32")
        senal = np.pad(senal, (0, FS * 8 - len(senal)))
    _ = sistema.clasificar_imagenes([imagen_clasificador(senal, evento.f_min, evento.f_max)])
    print("Clasificador verificado con un evento anotado de validation; el detector no produjo cajas operativas en el smoke test")
validado = True
print("Validación funcional completada")

100%|██████████| 253k/253k [00:00<00:00, 1.22MB/s]


100%|██████████| 6.04M/6.04M [00:00<00:00, 9.39MB/s]


100%|██████████| 1.69M/1.69M [00:00<00:00, 3.10MB/s]
[2026-11-22 02:41:53] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. Create predictor = native_model.inference(...) and use predictor.predict(...). For a checkpoint, first load native_model = RFDETR.from_checkpoint(...).


WABAD-CO::bre_20071210_054500: 307 cajas; 13 clasificadas


100%|██████████| 1.69M/1.69M [00:00<00:00, 4.39MB/s]


WABAD-CO::bre_20071210_054800: 260 cajas; 16 clasificadas
Validación funcional completada


## Congelar el protocolo

Esta celda registra las versiones de los modelos y del módulo. Si se modifica algún archivo después de generar resultados parciales, la ejecución se detendrá en vez de mezclar predicciones incompatibles. **No cambiar umbrales, pesos ni preprocesamiento después de comenzar `test`.**

In [6]:
assert validado
protocolo = {
    "detector": "RF-DETR Nano E5-v2", "clasificador": "Swin-T",
    "conf_min": 0.05, "conf_operativa": 0.50, "nms_iou": 0.50,
    "matching": {"iou": 0.5, "iomin": 0.5},
    "test_grabaciones": 152,
    "hash_detector": sha256(ARCHIVOS[0]),
    "hash_clasificador": sha256(ARCHIVOS[2]),
    "hash_codigo": sha256("/content/sistema_aves.py"),
    "hash_split": sha256(ARCHIVOS[3]),
}
ruta_protocolo = RESULTADOS / "protocolo.json"
if ruta_protocolo.exists():
    assert json.loads(ruta_protocolo.read_text(encoding="utf-8")) == protocolo, (
        "Protocolo diferente. No mezclar resultados de distintas versiones.")
else:
    ruta_protocolo.write_text(json.dumps(protocolo, indent=2), encoding="utf-8")
print("Protocolo congelado. Puede iniciarse test.")

Protocolo congelado. Puede iniciarse test.


## Inferencia del sistema en test

Procesamos todas las grabaciones del split oficial y guardamos **un CSV por grabación**. Cada CSV conserva las detecciones desde confianza 0,05; solamente las de confianza al menos 0,50 se clasifican. Si Colab se desconecta, podemos reanudar sin repetir los audios ya terminados. No usamos etiquetas reales para predecir.

In [7]:
test_audio = catalogo.loc[catalogo.split.eq("test")].sort_values("recording_uid").reset_index(drop=True)
assert len(test_audio) == 152
for fila in tqdm(test_audio.itertuples(index=False), total=len(test_audio), desc="Inferencia test"):
    nombre = hashlib.sha256(str(fila.recording_uid).encode()).hexdigest()[:20] + ".csv"
    destino = RESULTADOS / "predicciones_audio" / nombre
    if destino.is_file():
        previo = pd.read_csv(destino)
        assert set(previo.columns) == set(__import__("sistema_aves").COLUMNAS)
        assert previo.empty or previo.recording_uid.eq(fila.recording_uid).all()
        continue
    tabla = sistema.predecir_audio(ruta_audio(fila), fila.recording_uid,
                                  conf_min=protocolo["conf_min"],
                                  conf_operativa=protocolo["conf_operativa"])
    temporal = destino.with_suffix(".tmp")
    tabla.to_csv(temporal, index=False)
    temporal.replace(destino)

archivos_csv = sorted((RESULTADOS / "predicciones_audio").glob("*.csv"))
assert len(archivos_csv) == len(test_audio), "Inferencia incompleta"
pred = pd.concat((pd.read_csv(p) for p in archivos_csv), ignore_index=True)
assert set(pred.recording_uid).issubset(set(test_audio.recording_uid))
print("Grabaciones procesadas:", len(archivos_csv), "| Cajas conservadas:", len(pred))

Inferencia test:   0%|          | 0/152 [00:00<?, ?it/s]


  0%|          | 0.00/101M [00:00<?, ?B/s]
  1%|          | 1.00M/101M [00:00<00:35, 2.95MB/s]
  4%|▍         | 4.00M/101M [00:00<00:09, 10.4MB/s]
 10%|▉         | 10.0M/101M [00:00<00:03, 24.7MB/s]
 14%|█▍        | 14.0M/101M [00:00<00:04, 22.3MB/s]
 19%|█▉        | 19.0M/101M [00:00<00:02, 28.9MB/s]
 23%|██▎       | 23.0M/101M [00:01<00:02, 32.0MB/s]
 29%|██▉       | 29.0M/101M [00:01<00:01, 39.8MB/s]
 34%|███▍      | 34.0M/101M [00:01<00:01, 37.9MB/s]
 40%|███▉      | 40.0M/101M [00:01<00:01, 42.8MB/s]
 45%|████▍     | 45.0M/101M [00:01<00:01, 38.8MB/s]
 52%|█████▏    | 52.0M/101M [00:01<00:01, 45.0MB/s]
 59%|█████▊    | 59.0M/101M [00:01<00:01, 40.2MB/s]
 65%|██████▍   | 65.0M/101M [00:02<00:00, 40.1MB/s]
 72%|███████▏  | 72.0M/101M [00:02<00:00, 45.7MB/s]
 79%|███████▊  | 79.0M/101M [00:02<00:00, 50.2MB/s]
 85%|████████▌ | 86.0M/101M [00:02<00:00, 53.9MB/s]
 92%|█████████▏| 93.0M/101M [00:02<00:00, 58.6MB/s]
100%|██████████| 101M/101M [00:02<00:00, 40.3MB/s] 



  0%|          | 0.00/103M [00:00<?, ?B/s]
  1%|          | 1.00M/103M [00:00<00:36, 2.96MB/s]
  4%|▍         | 4.00M/103M [00:00<00:09, 10.4MB/s]
 11%|█         | 11.0M/103M [00:00<00:03, 26.6MB/s]
 15%|█▌        | 16.0M/103M [00:00<00:02, 33.3MB/s]
 22%|██▏       | 23.0M/103M [00:00<00:02, 39.1MB/s]
 29%|██▉       | 30.0M/103M [00:00<00:01, 46.1MB/s]
 34%|███▍      | 35.0M/103M [00:01<00:02, 31.9MB/s]
 39%|███▊      | 40.0M/103M [00:01<00:01, 35.6MB/s]
 44%|████▍     | 46.0M/103M [00:01<00:01, 41.5MB/s]
 49%|████▉     | 51.0M/103M [00:01<00:01, 42.8MB/s]
 55%|█████▌    | 57.0M/103M [00:01<00:01, 46.2MB/s]
 62%|██████▏   | 64.0M/103M [00:01<00:00, 48.4MB/s]
 69%|██████▊   | 71.0M/103M [00:01<00:00, 51.1MB/s]
 75%|███████▌  | 78.0M/103M [00:02<00:00, 54.4MB/s]
 82%|████████▏ | 85.0M/103M [00:02<00:00, 56.0MB/s]
 89%|████████▉ | 92.0M/103M [00:02<00:00, 57.8MB/s]
100%|██████████| 103M/103M [00:02<00:00, 43.5MB/s] 



  0%|          | 0.00/102M [00:00<?, ?B/s]
  1%|          | 1.00M/102M [00:00<00:42, 2.48MB/s]
  3%|▎         | 3.00M/102M [00:00<00:15, 6.89MB/s]
  8%|▊         | 8.00M/102M [00:00<00:05, 18.4MB/s]
 13%|█▎        | 13.0M/102M [00:00<00:04, 22.5MB/s]
 19%|█▊        | 19.0M/102M [00:00<00:02, 32.0MB/s]
 23%|██▎       | 23.0M/102M [00:01<00:02, 32.3MB/s]
 28%|██▊       | 28.0M/102M [00:01<00:02, 35.3MB/s]
 31%|███▏      | 32.0M/102M [00:01<00:02, 29.8MB/s]
 38%|███▊      | 39.0M/102M [00:01<00:01, 36.7MB/s]
 46%|████▌     | 47.0M/102M [00:01<00:01, 43.4MB/s]
 53%|█████▎    | 54.0M/102M [00:01<00:01, 44.9MB/s]
 60%|█████▉    | 61.0M/102M [00:01<00:00, 48.2MB/s]
 67%|██████▋   | 68.0M/102M [00:02<00:00, 51.6MB/s]
 73%|███████▎  | 74.0M/102M [00:02<00:00, 52.8MB/s]
 79%|███████▊  | 80.0M/102M [00:02<00:00, 51.4MB/s]
 84%|████████▎ | 85.0M/102M [00:02<00:00, 49.0MB/s]
 90%|█████████ | 92.0M/102M [00:02<00:00, 50.4MB/s]
100%|██████████| 102M/102M [00:02<00:00, 39.3MB/s] 



  0%|          | 0.00/109M [00:00<?, ?B/s]
  1%|          | 1.00M/109M [00:00<00:37, 2.97MB/s]
  4%|▎         | 4.00M/109M [00:00<00:10, 10.5MB/s]
 10%|█         | 11.0M/109M [00:00<00:04, 25.3MB/s]
 15%|█▍        | 16.0M/109M [00:00<00:03, 29.1MB/s]
 20%|██        | 22.0M/109M [00:00<00:02, 37.2MB/s]
 26%|██▌       | 28.0M/109M [00:00<00:01, 42.8MB/s]
 31%|███▏      | 34.0M/109M [00:01<00:01, 44.2MB/s]
 36%|███▌      | 39.0M/109M [00:01<00:01, 45.0MB/s]
 41%|████      | 44.0M/109M [00:01<00:02, 30.6MB/s]
 47%|████▋     | 51.0M/109M [00:01<00:01, 36.4MB/s]
 53%|█████▎    | 57.0M/109M [00:01<00:01, 38.9MB/s]
 59%|█████▉    | 64.0M/109M [00:01<00:01, 45.0MB/s]
 65%|██████▌   | 71.0M/109M [00:02<00:00, 49.8MB/s]
 72%|███████▏  | 78.0M/109M [00:02<00:00, 53.5MB/s]
 78%|███████▊  | 85.0M/109M [00:02<00:00, 57.1MB/s]
 85%|████████▍ | 92.0M/109M [00:02<00:00, 59.6MB/s]
 91%|█████████ | 99.0M/109M [00:02<00:00, 60.8MB/s]
100%|██████████| 109M/109M [00:02<00:00, 43.5MB/s]



  0%|          | 0.00/109M [00:00<?, ?B/s]
  1%|          | 1.00M/109M [00:00<00:38, 2.96MB/s]
  4%|▎         | 4.00M/109M [00:00<00:10, 10.4MB/s]
 10%|█         | 11.0M/109M [00:00<00:04, 23.7MB/s]
 17%|█▋        | 18.0M/109M [00:00<00:02, 33.4MB/s]
 23%|██▎       | 25.0M/109M [00:00<00:02, 43.1MB/s]
 28%|██▊       | 30.0M/109M [00:00<00:01, 45.0MB/s]
 32%|███▏      | 35.0M/109M [00:01<00:01, 39.3MB/s]
 39%|███▊      | 42.0M/109M [00:01<00:01, 45.9MB/s]
 45%|████▌     | 49.0M/109M [00:01<00:01, 50.9MB/s]
 52%|█████▏    | 56.0M/109M [00:01<00:01, 54.5MB/s]
 59%|█████▉    | 64.0M/109M [00:01<00:00, 59.0MB/s]
 64%|██████▍   | 70.0M/109M [00:01<00:00, 53.1MB/s]
 71%|███████   | 77.0M/109M [00:01<00:00, 55.9MB/s]
 78%|███████▊  | 85.0M/109M [00:02<00:00, 59.8MB/s]
 85%|████████▍ | 92.0M/109M [00:02<00:00, 60.9MB/s]
 91%|█████████ | 99.0M/109M [00:02<00:00, 61.7MB/s]
100%|██████████| 109M/109M [00:02<00:00, 48.0MB/s]


... (144 outputs left out)


  0%|          | 0.00/2.33M [00:00<?, ?B/s]
100%|██████████| 2.33M/2.33M [00:00<00:00, 5.73MB/s]



  0%|          | 0.00/2.36M [00:00<?, ?B/s]
100%|██████████| 2.36M/2.36M [00:00<00:00, 5.75MB/s]



  0%|          | 0.00/2.51M [00:00<?, ?B/s]
100%|██████████| 2.51M/2.51M [00:00<00:00, 6.04MB/s]


Grabaciones procesadas: 152 | Cajas conservadas: 303963


## Evaluación del detector

Comparamos las cajas de RF-DETR con **todas** las anotaciones de detección de `test` mediante asignación uno a uno, ordenada por confianza. Reportamos precisión, recall, F1 y falsas alarmas por hora con IoU e IoMin al 0,5. En una base con anotaciones incompletas, una falsa alarma significa «sin correspondencia anotada», no necesariamente ruido.

In [18]:
det = pd.read_csv(descargar(KAGGLE_ORIGEN, "DATASET_PROYECTO/metadata/detector_manifest.csv"), dtype={"event_id": str})
det = det.loc[det.recording_uid.isin(test_audio.recording_uid)].merge(
    test_audio[["recording_uid", "duration_sec"]], on="recording_uid", validate="many_to_one")
det["t_max"] = np.minimum(det.t_max, det.duration_sec)
det = det.loc[(det.t_max > det.t_min) & (det.f_max > det.f_min)].copy()
assert det.event_id.is_unique

def a_cajas(df):
    return np.column_stack([df.t_min, 1 - df.f_max / 16000,
                            df.t_max, 1 - df.f_min / 16000]).astype(np.float32)

def emparejar_detecciones(preds, gts, criterio="iou", minimo=0.50):
    from sistema_aves import iou as iou_uno
    a, b = a_cajas(preds), a_cajas(gts)
    usados, pares = set(), []
    for i in np.argsort(-preds.score_detector.to_numpy(), kind="stable"):
        disponibles = np.array([j for j in range(len(b)) if j not in usados], dtype=int)
        if not len(disponibles):
            continue
        if criterio == "iou":
            solape = iou_uno(a[i], b[disponibles])
        else:
            inter = np.maximum(np.minimum(a[i, 2:], b[disponibles, 2:]) -
                               np.maximum(a[i, :2], b[disponibles, :2]), 0).prod(axis=1)
            area_a = np.maximum(a[i, 2:] - a[i, :2], 0).prod()
            area_b = np.maximum(b[disponibles, 2:] - b[disponibles, :2], 0).prod(axis=1)
            minimo_area = np.minimum(area_a, area_b)
            solape = np.divide(inter, minimo_area, out=np.zeros_like(inter), where=minimo_area > 0)
        j = int(np.argmax(solape))
        if solape[j] >= minimo:
            elegido = int(disponibles[j]); usados.add(elegido)
            pares.append((int(i), elegido, float(solape[j])))
    return pares

pred_op = pred.loc[pred.score_detector >= protocolo["conf_operativa"]].copy()
horas = test_audio.duration_sec.sum() / 3600
resumen_det = []
for criterio in ("iou", "iomin"):
    tp = fp = fn = 0
    for uid in test_audio.recording_uid:
        p = pred_op.loc[pred_op.recording_uid.eq(uid)].reset_index(drop=True)
        g = det.loc[det.recording_uid.eq(uid)].reset_index(drop=True)
        aciertos = len(emparejar_detecciones(p, g, criterio))
        tp += aciertos; fp += len(p) - aciertos; fn += len(g) - aciertos
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    resumen_det.append({"criterio": criterio, "tp": tp, "fp": fp, "fn": fn,
                        "precision": precision, "recall": recall,
                        "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0,
                        "fp_h": fp / horas})
resumen_det = pd.DataFrame(resumen_det)
resumen_det.to_csv(RESULTADOS / "metricas_test_detector.csv", index=False)
display(resumen_det.round(4))

,criterio,tp,fp,fn,precision,recall,f1,fp_h
0,iou,2148,5192,5644,0.2926,0.2757,0.2839,246.7552
1,iomin,3745,3595,4047,0.5102,0.4806,0.4950,170.8561


## Evaluación individual de Swin-T

Utilizamos los **PNG de `test` centrados en las anotaciones reales** del dataset público del clasificador. El detector no interviene. Conservamos el mismo filtro `eligible_primary` y las mismas transformaciones de validación. Los TAR se leen directamente, sin extraerlos en Drive.

In [19]:
manifest = pd.read_csv(descargar(KAGGLE_CLAS, "manifest_test.csv"), dtype={"event_id": str})

assert len(manifest) == 4606 and manifest.event_id.is_unique
assert manifest.split.eq("test").all()
assert set(manifest.recording_uid).issubset(set(test_audio.recording_uid))

elegibles = (
    manifest.eligible_primary.astype(str)
    .str.lower()
    .map({"true": True, "false": False})
)

assert elegibles.notna().all()

clase_test = manifest.loc[elegibles].copy()

# Los espectrogramas están descomprimidos en Kaggle.
DIR_IMAGENES = Path(
    "/kaggle/input/clasificador-aves-espectrogramas-v1"
)

assert DIR_IMAGENES.is_dir(), "No existe el directorio de imágenes"

def ruta_png_test(fila):
    return (
        DIR_IMAGENES
        / Path(fila.archivo_tar).stem
        / fila.ruta_png
    )

# Verificar todas las imágenes antes de iniciar la inferencia.
faltantes = [
    str(ruta_png_test(fila))
    for fila in clase_test.itertuples(index=False)
    if not ruta_png_test(fila).is_file()
]

assert not faltantes, (
    f"Faltan {len(faltantes)} imágenes. "
    f"Ejemplos: {faltantes[:5]}"
)

print("Eventos elegibles:", len(clase_test))
print("Imágenes verificadas correctamente.")

# ============================================================
# Inferencia del clasificador
# ============================================================

for tar_nombre, grupo in tqdm(
    clase_test.groupby("archivo_tar", sort=True),
    desc="Clasificador test"
):
    destino = (
        RESULTADOS
        / "predicciones_clasificador"
        / f"{Path(tar_nombre).stem}.csv"
    )

    if destino.is_file():
        previo = pd.read_csv(
            destino,
            dtype={"event_id": str}
        )

        assert (
            previo.event_id.is_unique
            and set(previo.event_id) == set(grupo.event_id)
        ), "Cache incompatible"

        continue

    filas = []
    buffer, ids = [], []

    for fila in grupo.itertuples(index=False):

        ruta = ruta_png_test(fila)

        with Image.open(ruta) as png:
            imagen = png.convert("RGB")

        buffer.append(imagen)
        ids.append(fila.event_id)

        if len(buffer) == 16:
            y, conf = sistema.clasificar_imagenes(buffer)

            filas.extend(zip(ids, y, conf))

            buffer, ids = [], []

    if buffer:
        y, conf = sistema.clasificar_imagenes(buffer)
        filas.extend(zip(ids, y, conf))

    temporal = destino.with_suffix(".tmp")

    pd.DataFrame(
        filas,
        columns=["event_id", "pred_code", "confianza"]
    ).to_csv(temporal, index=False)

    temporal.replace(destino)

# ============================================================
# Consolidación y métricas
# ============================================================

archivos_cls = sorted(
    (RESULTADOS / "predicciones_clasificador").glob("*.csv")
)

assert len(archivos_cls) == clase_test.archivo_tar.nunique(), (
    "Faltan partes de test"
)

cls_preds = pd.concat(
    (
        pd.read_csv(p, dtype={"event_id": str})
        for p in archivos_cls
    ),
    ignore_index=True
)

assert (cls_preds.event_id.is_unique and set(cls_preds.event_id) == set(clase_test.event_id))

cls_resultado = clase_test.merge(cls_preds,on="event_id",validate="one_to_one")

real = cls_resultado.class_code.to_numpy(dtype=int)
predicho = cls_resultado.pred_code.to_numpy(dtype=int)

resumen_cls = pd.DataFrame([{
    "n_eventos": len(real),
    "macro_f1": f1_score(real, predicho, labels=list(range(31)), average="macro", zero_division=0),
    "weighted_f1": f1_score(real, predicho, labels=list(range(31)), average="weighted", zero_division=0),
    "accuracy": accuracy_score(real, predicho)
}])

resumen_cls.to_csv(RESULTADOS / "metricas_test_clasificador.csv",index=False)
cls_resultado.to_csv(RESULTADOS / "predicciones_test_clasificador.csv",index=False)
reporte = classification_report(real, predicho, labels=list(range(31)), output_dict=True, zero_division=0)

pd.DataFrame([
    {
        "codigo": i,
        "especie": codigos[i],
        **reporte[str(i)]
    }
    for i in range(31)
]).to_csv(RESULTADOS / "metricas_por_especie_test.csv", index=False)

por_fuente = []

for fuente, grupo in cls_resultado.groupby("dataset"):
    etiquetas_fuente = sorted(grupo.class_code.unique())
    por_fuente.append({
        "fuente": fuente,
        "n": len(grupo),
        "clases_presentes": len(etiquetas_fuente),
        "accuracy": accuracy_score(grupo.class_code, grupo.pred_code),
        "macro_f1_presentes": f1_score(grupo.class_code, grupo.pred_code, labels=etiquetas_fuente, average="macro", zero_division=0)
    })

pd.DataFrame(por_fuente).to_csv(RESULTADOS / "metricas_test_clasificador_por_fuente.csv", index=False)

display(resumen_cls.round(4))

Eventos elegibles: 4575
Imágenes verificadas correctamente.


Clasificador test:   0%|          | 0/2 [00:00<?, ?it/s]

,n_eventos,macro_f1,weighted_f1,accuracy
0,4575,0.6189,0.6931,0.6877


## Evaluación extremo a extremo

Reutilizamos las detecciones de `test` ya calculadas. Una predicción cuenta como **acierto** solo si corresponde geométricamente a un evento anotado y la especie coincide. Una clase equivocada produce un FP para la clase predicha y un FN para la clase real. Las detecciones asociadas a eventos sin etiqueta evaluable se registran aparte, porque **`OTHER_SPECIES` no equivale a especie desconocida**.

Usamos **IoU ≥ 0,5** como criterio geométrico principal, con la misma asignación uno a uno del detector; no se ajustan umbrales con `test`.

In [20]:
etiquetas = clase_test.set_index("event_id")["class_code"].astype(int).to_dict()
assert set(etiquetas).issubset(set(det.event_id)), "Eventos clasificables no encontrados en GT detector"
TP = np.zeros(31, dtype=int); FP = np.zeros(31, dtype=int); FN = np.zeros(31, dtype=int)
ignoradas = 0; detalle = []
for uid in tqdm(test_audio.recording_uid, desc="Evaluación integrada"):
    p = pred_op.loc[pred_op.recording_uid.eq(uid)].reset_index(drop=True)
    g = det.loc[det.recording_uid.eq(uid)].reset_index(drop=True)
    pares = emparejar_detecciones(p, g, "iou")
    indices_p = {i for i, _, _ in pares}
    ids_g = set()
    for i, j, solape in pares:
        fila_p, fila_g = p.iloc[i], g.iloc[j]
        event_id = fila_g.event_id
        ids_g.add(event_id)
        if event_id not in etiquetas:
            ignoradas += 1
            estado = "sin_etiqueta_evaluable"
            real_clase = np.nan
        else:
            real_clase = etiquetas[event_id]
            codigo = int(fila_p.pred_code)
            if codigo == real_clase:
                TP[codigo] += 1; estado = "correcto"
            else:
                FP[codigo] += 1; FN[real_clase] += 1; estado = "especie_incorrecta"
        detalle.append({"recording_uid": uid, "event_id": event_id, "estado": estado,
                        "real_code": real_clase, "pred_code": fila_p.pred_code,
                        "score_detector": fila_p.score_detector, "iou": solape,
                        "t_min": fila_p.t_min, "t_max": fila_p.t_max})
    for i, fila_p in p.iterrows():
        if i not in indices_p:
            codigo = int(fila_p.pred_code); FP[codigo] += 1
            detalle.append({"recording_uid": uid, "event_id": None, "estado": "sin_anotacion_coincidente",
                            "real_code": np.nan, "pred_code": codigo,
                            "score_detector": fila_p.score_detector, "iou": 0,
                            "t_min": fila_p.t_min, "t_max": fila_p.t_max})
    gt_clas = g.loc[g.event_id.isin(etiquetas)]
    for fila_g in gt_clas.itertuples(index=False):
        if fila_g.event_id not in ids_g:
            codigo = etiquetas[fila_g.event_id]; FN[codigo] += 1
            detalle.append({"recording_uid": uid, "event_id": fila_g.event_id, "estado": "no_detectado",
                            "real_code": codigo, "pred_code": np.nan,
                            "score_detector": np.nan, "iou": np.nan,
                            "t_min": fila_g.t_min, "t_max": fila_g.t_max})
assert int(TP.sum() + FN.sum()) == len(clase_test), "Contabilidad de eventos incompleta"
pres = np.divide(TP, TP + FP, out=np.zeros(31), where=(TP + FP) > 0)
rec = np.divide(TP, TP + FN, out=np.zeros(31), where=(TP + FN) > 0)
f1 = np.divide(2 * pres * rec, pres + rec, out=np.zeros(31), where=(pres + rec) > 0)
por_especie = pd.DataFrame({"codigo": range(31), "especie": [codigos[i] for i in range(31)],
    "tp": TP, "fp": FP, "fn": FN, "precision": pres, "recall": rec, "f1": f1})
por_especie.to_csv(RESULTADOS / "metricas_test_integrado_por_especie.csv", index=False)
pd.DataFrame(detalle).to_csv(RESULTADOS / "eventos_test_integrados.csv", index=False)
resumen_integrado = pd.DataFrame([{"n_eventos_evaluables": len(clase_test),
    "n_sin_etiqueta_evaluable": ignoradas, "tp": int(TP.sum()), "fp": int(FP.sum()), "fn": int(FN.sum()),
    "macro_f1": float(f1.mean()),
    "precision_micro": float(TP.sum() / (TP.sum() + FP.sum())) if TP.sum() + FP.sum() else 0,
    "recall_micro": float(TP.sum() / (TP.sum() + FN.sum())) if TP.sum() + FN.sum() else 0,
    "fp_h": float(FP.sum() / horas)}])
resumen_integrado.to_csv(RESULTADOS / "metricas_test_integrado.csv", index=False)
display(resumen_integrado.round(4))

Evaluación integrada:   0%|          | 0/152 [00:00<?, ?it/s]

,n_eventos_evaluables,n_sin_etiqueta_evaluable,tp,fp,fn,macro_f1,precision_micro,recall_micro,fp_h
0,4575,778,1089,5473,3486,0.2404,0.166,0.238,260.1101


## Exportación del sistema

Guardamos el módulo Python, los dos pesos seleccionados, sus configuraciones y el mapeo de 31 clases en una carpeta independiente. El notebook 13 importará `sistema_aves.py` y cargará el paquete sin entrenamiento ni necesidad de repetir esta evaluación. **No borres los checkpoints originales de Drive hasta verificar el paquete exportado.**

In [21]:
import shutil
archivos_export = {
    Path("/content/sistema_aves.py"): "sistema_aves.py",
    ARCHIVOS[0]: "modelo_detector.pth",
    ARCHIVOS[1]: "config_detector.json",
    ARCHIVOS[2]: "modelo_clasificador.pth",
}
for origen, nombre in archivos_export.items():
    destino = EXPORT / nombre
    if not destino.is_file() or sha256(destino) != sha256(origen):
        shutil.copy2(origen, destino)
    assert sha256(destino) == sha256(origen), f"Copia incorrecta: {nombre}"
(EXPORT / "class_mapping.json").write_text(json.dumps(mapa, indent=2), encoding="utf-8")
(EXPORT / "preprocessing_config.json").write_text(json.dumps(preproceso, indent=2), encoding="utf-8")
(EXPORT / "protocolo.json").write_text(json.dumps(protocolo, indent=2), encoding="utf-8")
(EXPORT / "requirements.txt").write_text(
    "rfdetr==1.12.0\nkagglehub==1.0.3\ntorch\ntorchvision\nsoundfile\nscipy\npandas\npillow\n", encoding="utf-8")
print("Paquete completo:", EXPORT)
print("Próximo: 13_Demostracion_Sistema.ipynb")

Paquete completo: /content/drive/MyDrive/Proyecto_Aves/Sistema_Aves_v1
Próximo: 13_Demostracion_Sistema.ipynb


## Conclusiones

La integración de RF-DETR Nano E5-v2 y Swin-T permitió construir y evaluar un sistema capaz de detectar, localizar y clasificar eventos acústicos de aves a partir de grabaciones completas.

La evaluación sobre 152 grabaciones del conjunto de test permitió obtener los siguientes hallazgos:

- **Detección:** RF-DETR alcanzó un F1 de 0.2839 con IoU@0.50 y de 0.4950 con IoMin@0.50, evidenciando dificultades para delimitar con precisión los eventos acústicos.
- **Clasificación:** Swin-T obtuvo un Macro-F1 de 0.6189 y una exactitud del 68.77 % sobre 4575 eventos anotados, mostrando un rendimiento superior cuando recibe regiones de referencia.
- **Sistema integrado:** el Macro-F1 disminuyó a 0.2404, con un recall micro de 0.2380 y aproximadamente 260 falsos positivos por hora, lo que evidencia las limitaciones del sistema completo.

En conjunto, los resultados muestran que **el rendimiento individual de los modelos no garantiza un desempeño equivalente al integrarlos**. La detección constituye una limitación importante, pues los errores de localización y las omisiones afectan directamente la clasificación posterior.

Como trabajo futuro, se propone mejorar la capacidad de detección, analizar los errores por especie y fuente de datos, y optimizar los umbrales operativos para reducir falsas alarmas sin comprometer excesivamente el recall.